In [39]:
pip install scipy arviz

Note: you may need to restart the kernel to use updated packages.


In [40]:
import numpy as np
import pandas as pd
import scipy as sp
import arviz as az

In [41]:
import pickle

ruta = "/Users/Usuario/Downloads/Anfora/Modelos/M1_logistica.pkl"

with open(ruta, "rb") as f:
    M1 = pickle.load(f)

print(type(M1))

<class 'dict'>


In [47]:
prediccion = M1.predict()

AttributeError: 'dict' object has no attribute 'predict'

In [ ]:
import pandas as pd

df = pd.read_json("/Users/Usuario/Downloads/Anfora/Operaciones/metricas_T0_validacion.json", orient="index")


df.head()

,descripcion,features,threshold,T0_train,T0_validacion
M1_logistica,Regresión logística lineal con 3 features (mon...,"[monto_log, canal, dispositivo]",0.14,"{'n': 420460, 'n_fraude': 3355, 'threshold': 0...","{'n': 180832, 'n_fraude': 1448, 'threshold': 0..."
M2_random_forest,Random Forest con 5 features (incluye hora_bin...,"[monto_log, secuencia_24h, canal, dispositivo,...",0.11,"{'n': 420460, 'n_fraude': 3355, 'threshold': 0...","{'n': 180832, 'n_fraude': 1448, 'threshold': 0..."
M3_gbm_completo,Gradient Boosting completo con 6 features incl...,"[monto_log, secuencia_24h, terminal_risk_score...",0.46,"{'n': 420460, 'n_fraude': 3355, 'threshold': 0...","{'n': 180832, 'n_fraude': 1448, 'threshold': 0..."
M4_gbm_moderado,Gradient Boosting moderado con 5 features (sin...,"[monto_log, secuencia_24h, canal, dispositivo,...",0.11,"{'n': 420460, 'n_fraude': 3355, 'threshold': 0...","{'n': 180832, 'n_fraude': 1448, 'threshold': 0..."
M5_naive_bayes,Naive Bayes Gaussiano con solo 2 features estr...,"[monto_log, secuencia_24h]",0.11,"{'n': 420460, 'n_fraude': 3355, 'threshold': 0...","{'n': 180832, 'n_fraude': 1448, 'threshold': 0..."


In [ ]:
# Convertir una columna con un diccionario a columnas separadas: 

Metricas_T0_Val = df['T0_validacion'].apply(pd.Series)

In [48]:
Metricas_T0_Val

,n,n_fraude,threshold,TP,FP,FN,TN,precision,recall,f1,accuracy,auc_roc,auc_pr,f1_score_recalculada
M1_logistica,180832.0,1448.0,0.14,347.0,1065.0,1101.0,178319.0,0.245751,0.239641,0.242657,0.988022,0.912089,0.171716,0.242657
M2_random_forest,180832.0,1448.0,0.11,377.0,1272.0,1071.0,178112.0,0.228623,0.260359,0.243461,0.987043,0.895831,0.161155,0.243461
M3_gbm_completo,180832.0,1448.0,0.46,1176.0,170.0,272.0,179214.0,0.873700,0.812155,0.841804,0.997556,0.996436,0.854691,0.841804
M4_gbm_moderado,180832.0,1448.0,0.11,395.0,1507.0,1053.0,177877.0,0.207676,0.272790,0.235821,0.985843,0.911858,0.148068,0.235821
M5_naive_bayes,180832.0,1448.0,0.11,330.0,1312.0,1118.0,178072.0,0.200974,0.227901,0.213592,0.986562,0.896472,0.137176,0.213592


In [44]:
# sanity check: verificar que las métricas se correspondan con las del modelo M1:

def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


In [ ]:
# Quiero agregar una coluna con el f1_score recalculado a partir de precision y recall para cada fila

Metricas_T0_Val['f1_score_recalculada'] = Metricas_T0_Val.apply(lambda row: f1_score(row['precision'], row['recall']), axis=1)

In [46]:
Metricas_T0_Val

,n,n_fraude,threshold,TP,FP,FN,TN,precision,recall,f1,accuracy,auc_roc,auc_pr,f1_score_recalculada
M1_logistica,180832.0,1448.0,0.14,347.0,1065.0,1101.0,178319.0,0.245751,0.239641,0.242657,0.988022,0.912089,0.171716,0.242657
M2_random_forest,180832.0,1448.0,0.11,377.0,1272.0,1071.0,178112.0,0.228623,0.260359,0.243461,0.987043,0.895831,0.161155,0.243461
M3_gbm_completo,180832.0,1448.0,0.46,1176.0,170.0,272.0,179214.0,0.873700,0.812155,0.841804,0.997556,0.996436,0.854691,0.841804
M4_gbm_moderado,180832.0,1448.0,0.11,395.0,1507.0,1053.0,177877.0,0.207676,0.272790,0.235821,0.985843,0.911858,0.148068,0.235821
M5_naive_bayes,180832.0,1448.0,0.11,330.0,1312.0,1118.0,178072.0,0.200974,0.227901,0.213592,0.986562,0.896472,0.137176,0.213592


### Paso 2. Primer posterior Beta-Binomial sobre un solo modelo
Tomar M3 y la validación T0 entera. 

#### Construir dos posteriores Beta-Binomial conjugadas: 
    una sobre la precisión, p | datos ∼ Beta(1 + TP, 1 + FP), 
    y otra sobre la sensibilidad, r | datos ∼ Beta(1 + TP, 1 + FN). 
    
    Con prior uniforme Beta(1, 1), esa es la actualización conjugada estándar.

Reportar mediana y HPDI 95 % de cada una.

Lo importante de este paso no es el número en sí sino tener dos cosas claras: 
(i) por qué la elección Beta-Binomial es natural — el dato es una secuencia de éxitos y fracasos, y la
posterior conjugada es Beta; 

(ii) qué supone la posterior — que precisión y sensibilidad son?

In [ ]:

from scipy.stats import beta

# Datos
k = 7      # éxitos
n = 10     # intentos

# Prior uniforme Beta(1,1)
alpha_prior = 1
beta_prior = 1

# Posterior
alpha_post = alpha_prior + k
beta_post = beta_prior + (n - k)

# Media posterior
media_posterior = alpha_post / (alpha_post + beta_post)

print(alpha_post, beta_post)
print(media_posterior)


8 4
0.6666666666666666
